# Viz — Datashader

Aggregate a large scatter to a raster, then save PNG. Requires ``pip install datashader``.

**Datashader aggregation → PNG**

Bins points into a **1800×1800** canvas (doubled resolution), aggregates mean `log1p(degree)` per pixel, shades with a perceptual colormap (`colorcet` if installed, else a short fallback ramp), and saves a single raster under `cache/figures/`. **Interpretation:** bright regions are dense in the plane *and* have high average degree at the chosen resolution; sparse tails disappear by design.

In [ ]:
from pathlib import Path

import datashader as ds
import datashader.transfer_functions as tf
import numpy as np

try:
    import colorcet as cc

    _cmap = cc.fire
except Exception:
    _cmap = ["#000000", "#471063", "#f7d13d", "#ffffff"]
import pandas as pd

import dmercator_io as dm

merged = dm.load_merged_parquet(Path("cache") / "merged.parquet")
u, v = dm.stereo_disk_north(merged)
df = pd.DataFrame({"u": u, "v": v, "d": np.log1p(merged["degree"].to_numpy())})
clip = 10.0
df = df[(df["u"].abs() < clip) & (df["v"].abs() < clip)]

cvs = ds.Canvas(plot_width=1800, plot_height=1800, x_range=(-clip, clip), y_range=(-clip, clip))
agg = cvs.points(df, "u", "v", ds.mean("d"))
img = tf.shade(agg, cmap=_cmap, how="eq_hist")
out = Path("cache") / "figures"
out.mkdir(parents=True, exist_ok=True)
png_path = out / "datashader_stereo_mean_logdeg.png"
img.to_pil().save(png_path)
print("Wrote", png_path.resolve())
img
